The purpose of this notebook is to demonstrate the deep learning training phenomenon of grokking.  To demonstrate this, we follow the experiment performed in https://arxiv.org/pdf/2205.10343. This includes doing the following:

1.   We reduce the size of the training set from 50k to 1k samples (by taking a random subset)
2.   We increase the scale of the
weight initialization distribution (by multiplying the initial weights, sampled with Kaiming uniform
initialization, by a constant > 1).

## Setup

In [1]:
### imports
import torch
import torch.nn as nn
import torch.nn.init as init
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import wandb
from torchvision import datasets, transforms
import pandas as pd
import os
from torchvision.transforms import ToTensor
from sklearn.manifold import TSNE
import torch.nn.functional as F
import math
import copy
from collections import deque
from typing import Dict, Optional, Literal

# Import BayesNet components
import sys
sys.path.insert(0, '/home/phancock/Grokking/grokking_mnist')
from Bayesian_models.BayesNet import KFCALLAWrapper, CLIPZeroShotClassifier

#Ensure right device
def get_device():
    """Get the appropriate device for model inference."""
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()

print(f"Using device: {DEVICE}")

Using device: cuda


### Set up Model architectures

In [2]:
### Set up SimpleMLP architecture with MNIST dataset
def scaled_kaiming_init(module, scale=2.0):
    if isinstance(module, nn.Linear):
        init.kaiming_uniform_(module.weight, nonlinearity='relu')
        module.weight.data *= scale

        if module.bias is not None:
            nn.init.zeros_(module.bias)

class SimpleMLP(nn.Module):
    def __init__(self, input_features = 28*28, hidden_units=200, output_classes=10):
        super().__init__()
        self.stack = nn.Sequential(
            nn.Linear(input_features, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, output_classes)
        )
        self.last_layer = self.stack[-1]

    def forward(self, x):
        x = x.view(x.shape[0], -1)
        x = self.stack(x)
        return x

    def feat_nograd_forward(self, x):
        x = x.view(x.shape[0], -1)
        with torch.no_grad():
            for layer in self.stack[:-1]:  # Exclude the last layer
                x = layer(x)
        feat = x
        x = self.stack[-1](x)  # Pass through the last layer
        return x, feat

    # Set up data
MNIST_train_data = datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=ToTensor(),
)

np.random.seed(42) # for reproducibility
MNIST_train_data_subset = torch.utils.data.Subset(MNIST_train_data, np.random.choice(len(MNIST_train_data), 1000))

MNIST_val_data = datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=ToTensor()
)

# Create a wrapped dataset that includes the original data and targets, as well as an index for each sample to facilitate loss caching and retrieval for RHOLoss
class wrapped_dataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        try:
            self.targets = dataset.targets
        except:
            self.targets = dataset.dataset.targets
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        return {
            'input': self.dataset[index][0],
            'target': self.dataset[index][1],
            'index': index
        }

# Set up train dataloader for RHOLoss method
MNIST_dataloader = torch.utils.data.DataLoader(wrapped_dataset(MNIST_train_data_subset), batch_size=128, shuffle=False)

In [3]:
### Set up Transformer architecture with ModSubtractDataset
import sys
sys.path.insert(0, '../grokking_algorithmic/grokk_replica')

from datasets import ModSubtractDataset
from grokk_model import GrokkModel

# Set up the ModSubtractDataset with modular arithmetic p=97, 50% training split
p = 96
frac_train = 0.4
dataset = ModSubtractDataset(p=p, frac_train=frac_train)

# Configure the Transformer model
transformer_config = {
    'max_length': 5,  # For equations like "a o b = c"
    'heads': 4,
    'hidden_dim': 128,
    'attn_dim': 32,
    'intermediate_dim': 512,
    'num_blocks': 2,
    'block_repeats': 1,
    'dropout': 0.1,
    'pre_norm': True
}

# Create GrokkModel instance
vocab_size = dataset.n_vocab
output_size = dataset.n_out

transformer_model = GrokkModel(
    transformer_config=transformer_config,
    vocab_size=vocab_size,
    output_size=output_size,
    device=DEVICE
).to(DEVICE)

print(f"Dataset: ModMultDataset with p={p}")
print(f"Vocab size: {vocab_size}, Output size: {output_size}")
print(f"transformer_model created with {sum(p.numel() for p in transformer_model.parameters())} parameters")

# Create a wrapper class for ModSubtractDataset to be compatible with the train function
class AlgorithmicDatasetWrapper(torch.utils.data.Dataset):
    """Wrapper for ModSubtractDataset that provides the expected data format for training."""
    def __init__(self, dataset, pair_indices):
        self.dataset = dataset
        self.pair_indices = pair_indices  # indices into the dataset

    def __len__(self):
        return len(self.pair_indices)

    def __getitem__(self, idx):
        pair_idx = self.pair_indices[idx]
        input_seq, target, _ = self.dataset.fetch_example(pair_idx)
        return {
            'input': torch.tensor(input_seq, dtype=torch.long),
            'target': torch.tensor(target, dtype=torch.long),
            'index': idx
        }

# Create training and validation datasets
train_indices = np.array(dataset.train_pairs)
val_indices = np.array(dataset.val_pairs)

algorithmic_train_dataset = AlgorithmicDatasetWrapper(dataset, train_indices)
algorithmic_val_dataset = AlgorithmicDatasetWrapper(dataset, val_indices)

Dataset: ModMultDataset with p=96
Vocab size: 98, Output size: 96
transformer_model created with 422368 parameters


## Helper Functions

In [4]:
### Plotting functions
def ema(data, alpha=0.2):
    smoothed = []
    s = data[0]
    for point in data:
        s = alpha * point + (1 - alpha) * s
        smoothed.append(s)
    return smoothed

def plot_accuracies(train_accuracies, val_accuracies, log_interval, optimization_steps, include_ema=True, save_plot=False, plot_name="accuracy_plot.png"):
    # Plot accuracies
    x = range(0, optimization_steps, log_interval)
    plt.plot(x, train_accuracies, color='r')
    plt.plot(x, val_accuracies, color='g')
    if include_ema:
        train_smoothed = ema(train_accuracies, alpha=0.2)
        val_smoothed   = ema(val_accuracies, alpha=0.2)
        plt.plot(x, train_smoothed, color='y')
        plt.plot(x, val_smoothed, color='b')
        plt.legend(['Train', 'Val', "EMA_Train", "EMA_Val"])
    else:
        plt.legend(['Train', 'Val'])
    plt.xlabel('Optimization Steps')
    plt.ylabel('Accuracy')
    plt.title('Accuracy')
    plt.xscale('log')
    if save_plot:
        plt.savefig(plot_name)
    plt.show()

def plot_losses(train_losses, val_losses, log_interval, optimization_steps, save_plot=False, plot_name="loss_plot.png"):
    # Plot losses
    x = range(0, optimization_steps, log_interval)
    plt.plot(x, train_losses, color='r')
    plt.plot(x, val_losses, color='g')
    plt.xlabel('Optimization Steps')
    plt.ylabel('Loss')
    plt.title('Loss')
    plt.xscale('log')
    plt.legend(['Train', 'Val'])
    if save_plot:
        plt.savefig(plot_name)
    plt.show()

### Grokfast
def gradfilter_ema(
    m: nn.Module,
    grads: Optional[Dict[str, torch.Tensor]] = None,
    alpha: float = 0.8,
    lamb: float = 0.1,
) -> Dict[str, torch.Tensor]:
    if grads is None:
        grads = {n: p.grad.data.detach() for n, p in m.named_parameters() if p.requires_grad}

    for n, p in m.named_parameters():
        if p.requires_grad:
            grads[n] = grads[n] * alpha + p.grad.data.detach() * (1 - alpha)
            p.grad.data = p.grad.data + grads[n] * lamb

    return grads


# Grokfast-MA
def gradfilter_ma(
    m: nn.Module,
    grads: Optional[Dict[str, deque]] = None,
    window_size: int = 128,
    lamb: float = 5.0,
    filter_type: Literal['mean', 'sum'] = 'mean',
    warmup: bool = True,
    trigger: bool = False,
) -> Dict[str, deque]:
    if grads is None:
        grads = {n: deque(maxlen=window_size) for n, p in m.named_parameters() if p.requires_grad}

    for n, p in m.named_parameters():
        if p.requires_grad:
            grads[n].append(p.grad.data.detach())

            if not warmup or len(grads[n]) == window_size and not trigger:
                if filter_type == "mean":
                    avg = sum(grads[n]) / len(grads[n])
                elif filter_type == "sum":
                    avg = sum(grads[n])
                else:
                    raise ValueError(f"Unrecognized filter_type {filter_type}")
                p.grad.data = p.grad.data + avg * lamb

    return grads

### Selection Methods
# DivBS Selection
class DivBS:
    def __init__(self, model, is_transformer=False):
        self.model = model
        self.is_transformer = is_transformer

    def calc_grad(self, model, inputs, targets, reduce_dim=False):
        """Compute diversity-based gradients for sample selection.

        Works with both transformer and MLP architectures by using output
        representations rather than explicit parameter gradients.
        """
        model = model.module if isinstance(model, torch.nn.DataParallel) else model
        model.eval()

        with torch.no_grad():
            # Get model outputs (features for diversity computation)
            if self.is_transformer:
                outputs, _ = model(inputs)
                outputs = outputs[:, -1, :].float()  # Use last token output
            else:
                outputs = model(inputs)

            # Use softmax probabilities as diversity features
            features = torch.softmax(outputs, dim=-1)

            # Compute per-sample loss for gradient direction
            per_sample_loss = torch.nn.functional.cross_entropy(
                outputs, targets, reduction='none'
            )

        model.train()

        # Create gradient matrix: (batch_size, feature_dim)
        # Row i contains the loss-weighted feature representation of sample i
        grad = features * per_sample_loss.unsqueeze(1)

        if reduce_dim is not False and reduce_dim > 0:
            dim = grad.shape[1]
            dim_reduced = max(1, dim // reduce_dim)
            index = np.random.choice(dim, dim_reduced, replace=False)
            grad = grad[:, index]

        grad_mean = grad.mean(dim=0)
        return grad_mean, grad

    def greedy_selection(self, grad_mean, grad, number_to_select):
        residual = grad_mean.unsqueeze(-1) if grad_mean.dim() == 1 else grad_mean
        index_selected = []
        D = grad.t()
        selected_element = []
        for i in range(number_to_select):
            correlations = torch.abs(torch.matmul(D.t(), residual))
            
            # Numerical stability checks
            # Replace NaN with small positive value
            correlations = torch.where(torch.isnan(correlations), torch.full_like(correlations, 1e-8), correlations)
            # Replace Inf with maximum finite value
            correlations = torch.clamp(correlations, max=torch.finfo(correlations.dtype).max)
            # Clamp negative values to 0
            correlations = torch.clamp(correlations, min=0.0)
            # Add epsilon to prevent all-zero probabilities
            correlations = correlations + 1e-8
            # Normalize to sum to 1 (required for multinomial)
            correlations = correlations / (correlations.sum() + 1e-8)
            
            try:
                idx = torch.multinomial(correlations.squeeze(), 1)
            except:
                break
            index_selected.append(idx.item())
            if len(selected_element) > 0:
                selected_element_matrix = torch.cat(selected_element, dim=1)
                D_selected = D[:, idx] - torch.matmul(selected_element_matrix,torch.matmul(selected_element_matrix.t(), D[:, idx]))
            else:
                D_selected = D[:, idx]
            # Normalize with safety check for zero norm
            norm_val = torch.norm(D_selected)
            if norm_val > 1e-8:
                D_selected = D_selected / norm_val
            selected_element.append(D_selected)
            residual = residual - torch.matmul(D_selected.t(), residual) * D_selected

        if len(selected_element) < number_to_select:
            num_random = number_to_select - len(selected_element)
            remaining_indices = list(set(range(grad.shape[0])) - set(index_selected))
            random_indices = np.random.choice(remaining_indices, num_random, replace=False)
            index_selected.extend(random_indices)

        return index_selected

    def select_points(self, inputs, targets, fraction=0.25):
        grad_mean, grad = self.calc_grad(self.model, inputs, targets)
        selected_num_samples = int(inputs.shape[0] * fraction)
        indices = self.greedy_selection(grad_mean, grad, selected_num_samples)
        return indices

# RHOLoss Selection
class RHOLoss:
    def __init__(self, model, train_loader, is_transformer, teacher_checkpoint_path=None, teacher_model=None, uniform_epochs=0):
        self.model = model
        self.is_transformer = hasattr(model, 'transformer_config') or hasattr(model, 'transformer')
        self.train_loader = train_loader
        self.uniform_epochs = uniform_epochs
        self.device = DEVICE
        self.train_dataset = getattr(train_loader, "dataset", None)
        self.teacher_checkpoint_path = teacher_checkpoint_path

        self.teacher_model = self._build_teacher_model(teacher_model, teacher_checkpoint_path)
        self.teacher_model.to(self.device)
        self.teacher_model.eval()

        for param in self.teacher_model.parameters():
            param.requires_grad_(False)

        if self.train_dataset is not None:
            self.precompute_losses()

    def _extract_state_dict(self, checkpoint):
        if isinstance(checkpoint, dict):
            candidate_keys = ["model_state_dict", "state_dict", "model"]
            for key in candidate_keys:
                if key in checkpoint and isinstance(checkpoint[key], dict):
                    return checkpoint[key]

            if checkpoint and all(torch.is_tensor(value) for value in checkpoint.values()):
                return checkpoint

        raise ValueError(
            "Unsupported teacher checkpoint format. Expected a raw state_dict like the files produced by train(..., save_checkpoints=True)."
        )

    def _build_teacher_model(self, teacher_model=None, teacher_checkpoint_path=None):
        if teacher_model is not None:
            return teacher_model

        if teacher_checkpoint_path is not None:
            base_model = self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model
            teacher_model = copy.deepcopy(base_model)
            checkpoint = torch.load(teacher_checkpoint_path, map_location='cpu')
            state_dict = self._extract_state_dict(checkpoint)
            teacher_model.load_state_dict(state_dict)
            return teacher_model

        if self.is_transformer:
            raise ValueError(
                "Transformer RHOLoss requires teacher_checkpoint_path or an instantiated teacher_model. Pass the checkpoint saved by train(..., save_checkpoints=True)."
            )

        return CLIPZeroShotClassifier(
            classnames=[str(i) for i in range(10)],
            template=["A photo of the digit {}"],
            dataset="MNIST",
            arch="RN50",
            tau=4.0,
        )

    def precompute_losses(self):
        """Precompute irreducible losses for the training dataset using the holdout model."""
        if self.is_transformer:
            losses_tensor = torch.zeros(len(self.train_dataset))
            with torch.no_grad():
                for datas in self.train_loader:
                    inputs = datas['input'].to(self.device)
                    targets = datas['target'].to(self.device)
                    indexes = datas['index']
                    predictions, _ = self.teacher_model(inputs)
                    loss = F.cross_entropy(predictions[:, -1, :], targets, reduction='none')
                    losses_tensor[indexes] = loss.float().cpu()
        else:
            losses_tensor = torch.zeros(len(self.train_dataset))
            with torch.no_grad():
                for datas in self.train_loader:
                    inputs = datas['input'].to(self.device)
                    targets = datas['target'].to(self.device)
                    indexes = datas['index']
                    outputs = self.teacher_model(inputs)
                    targets_oh = torch.nn.functional.one_hot(targets, num_classes=10).float()
                    criterion = torch.nn.MSELoss(reduction='none')
                    loss = criterion(outputs, targets_oh).mean(dim=1)
                    losses_tensor[indexes] = loss.float().cpu()

        self.train_dataset.irreducible_loss_cache = losses_tensor
        print(f"Cached irreducible losses for {len(losses_tensor)} samples in dataset.")

    def select_points(self, inputs, targets, indexes, num_to_select, epoch):
        """Select sub-batch with highest reducible loss."""
        self.model.eval()

        if self.is_transformer:
            with torch.no_grad():
                predictions, _ = self.model(inputs)
                total_loss = F.cross_entropy(predictions[:, -1, :], targets, reduction='none')
        else:
            with torch.no_grad():
                criterion = torch.nn.MSELoss(reduction='none')
                targets_oh = torch.nn.functional.one_hot(targets, num_classes=10).float()
                total_loss = criterion(self.model(inputs), targets_oh).mean(dim=1)

        irreducible_loss = self.train_dataset.irreducible_loss_cache[indexes].to(total_loss.device)
        reducible_loss = total_loss - irreducible_loss

        num_to_select = min(num_to_select, len(inputs))
        _, indices = torch.topk(reducible_loss, num_to_select, largest=True, sorted=False)

        if epoch < self.uniform_epochs:
            indices = torch.randperm(len(inputs))[:num_to_select]

        self.model.train()
        return indices.cpu().numpy()

# Bayesian Selection with KFCA and optional CLIP guidance
class Bayesian:
    def __init__(
        self,
        model,
        train_dataset,
        train_loader,
        teacher_checkpoint_path=None,
        teacher_model=None,
        num_effective_data=200,
        prior_precision=10,
        n_f_samples=256,
        laplace_momentum=0.99,
    ):
        self.device = DEVICE
        self.train_dataset = train_dataset
        self.is_transformer = hasattr(model, 'transformer_config') or hasattr(model, 'transformer')
        self.num_effective_data = num_effective_data
        self.prior_precision = prior_precision
        self.n_f_samples = n_f_samples
        self.laplace_momentum = laplace_momentum
        self.train_loader = train_loader

        self.model = KFCALLAWrapper(
            net=model,
            num_effective_data=self.num_effective_data,
            prior_precision=self.prior_precision,
            n_f_samples=self.n_f_samples,
            last_layer_name="last_layer",
            momentum=self.laplace_momentum,
        )
        self.model.to(self.device)
        self.model.eval()
        self.teacher_model = self._build_teacher_model(teacher_model, teacher_checkpoint_path)
        if self.teacher_model is not None:
            self.teacher_model.to(self.device)
            self.teacher_model.eval()
        self.clip_model = self.teacher_model if not self.is_transformer else None
        self.use_clip = self.clip_model is not None
        self.use_teacher = self.teacher_model is not None

        self.alpha = 0.2 if self.use_teacher else 1.0
        self.adaptive_alpha = False

        if self.train_dataset is not None and self.use_teacher and self.train_loader is not None:
            self.precompute_losses()

    def _extract_state_dict(self, checkpoint):
        if isinstance(checkpoint, dict):
            candidate_keys = ["model_state_dict", "state_dict", "model"]
            for key in candidate_keys:
                if key in checkpoint and isinstance(checkpoint[key], dict):
                    return checkpoint[key]

            if checkpoint and all(torch.is_tensor(value) for value in checkpoint.values()):
                return checkpoint

        raise ValueError(
            "Unsupported teacher checkpoint format. Expected a raw state_dict like the files produced by train(..., save_checkpoints=True)."
        )

    def _build_teacher_model(self, teacher_model=None, teacher_checkpoint_path=None):
        if teacher_model is not None:
            return teacher_model

        if teacher_checkpoint_path is not None:
            base_model = self.model.net.module if isinstance(self.model.net, torch.nn.DataParallel) else self.model.net
            teacher_model = copy.deepcopy(base_model)
            checkpoint = torch.load(teacher_checkpoint_path, map_location='cpu')
            state_dict = self._extract_state_dict(checkpoint)
            teacher_model.load_state_dict(state_dict)
            return teacher_model

        if self.is_transformer:
            return None

        return CLIPZeroShotClassifier(
            classnames=[str(i) for i in range(10)],
            template=["A photo of the digit {}"],
            dataset="MNIST",
            arch="RN50",
            tau=4.0,
        )

    def precompute_losses(self):
        """Cache teacher or CLIP losses for every example in the training dataset."""
        if self.train_dataset is None or self.teacher_model is None or self.train_loader is None:
            return

        losses_tensor = torch.zeros(len(self.train_dataset))
        with torch.no_grad():
            for datas in self.train_loader:
                inputs = datas["input"].to(self.device)
                targets = datas["target"].to(self.device)
                indexes = datas["index"]

                outputs = self.teacher_model(inputs)
                if self.is_transformer:
                    if isinstance(outputs, tuple):
                        outputs = outputs[0]
                    loss = F.cross_entropy(outputs[:, -1, :], targets, reduction="none")
                else:
                    loss = -F.cross_entropy(outputs, targets, reduction="none")

                losses_tensor[indexes] = loss.cpu().float()
        self.train_dataset.clip_loss_cache = losses_tensor
        print(f"Cached teacher/CLIP losses for {len(losses_tensor)} samples in dataset.")

    def _surrogate_loss(self, logits, targets):
        if self.is_transformer:
            return F.cross_entropy(logits, targets, reduction="none")

        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=10).float().to(logits.device)
        return F.mse_loss(logits, targets_one_hot, reduction="none").mean(dim=1)

    def select_points(self, inputs, targets, indexes, num_to_select):
        self.model.train()
        with torch.no_grad():
            _ = self.model(inputs, targets=targets)

        self.model.eval()
        with torch.no_grad():
            f_samples, outputs, stds, L_U_T_inverse = self.model(
                inputs, selection_pass=True
            )

            device = f_samples.device
            targets = targets.to(device)

            f_samples_flat = f_samples.flatten(0, 1)
            targets_repeated = targets.repeat_interleave(f_samples.shape[1], dim=0)
            first_term = self._surrogate_loss(f_samples_flat, targets_repeated)
            first_term = first_term.view(f_samples.shape[0], f_samples.shape[1]).mean(1)

            second_term = torch.zeros_like(first_term)
            if (
                self.use_teacher
                and self.train_dataset is not None
                and indexes is not None
                and hasattr(self.train_dataset, "clip_loss_cache")
            ):
                if isinstance(indexes, torch.Tensor):
                    index_tensor = indexes.detach().cpu().long()
                else:
                    index_tensor = torch.as_tensor(indexes, dtype=torch.long)
                second_term = self.train_dataset.clip_loss_cache[index_tensor].to(device)

                if self.adaptive_alpha:
                    bayes_loss = first_term.mean().abs() + 1e-8
                    clip_loss = second_term.mean().abs() + 1e-8
                    self.alpha = (clip_loss / (bayes_loss + clip_loss)).item()

            mean_probs = f_samples.softmax(-1).mean(1).clamp_min(1e-8)
            predictive_entropy = -(mean_probs * mean_probs.log()).sum(dim=1)
            hardness = F.cross_entropy(mean_probs.log(), targets, reduction="none")

            select_obj = (
                self.alpha * first_term
                + (1 - self.alpha) * second_term
                + predictive_entropy
                + hardness
            )
            num_to_select = min(num_to_select, len(inputs))
            _, index_selected = torch.topk(select_obj, num_to_select)

        return index_selected.cpu().numpy()

#### Training and Evaluation

In [5]:
# Universal train function for both MLP and Transformer models with all batch selection methods
def train(model, optimizer, criterion, train_loader, val_loader, optimization_steps,
          log_interval, save_checkpoints = False, save_interval=10000, selection_fn="full",
          fraction=1, use_wandb = False, grokFast = False, teacher_checkpoint_path=None,
          teacher_model=None, scheduler=None, lr = 1e-3, weight_decay = 0, seed = 42):
    model.to(DEVICE)

    # Detect model type
    is_transformer = hasattr(model, 'transformer_config') or hasattr(model, 'transformer')

    train_accuracies = []
    train_losses = []
    val_accuracies = []
    val_losses = []
    step = 0

    # Set up selector based on model type
    selector = set_up_selector(
        model,
        selection_fn,
        is_transformer,
        train_loader=train_loader,
        teacher_checkpoint_path=teacher_checkpoint_path,
        teacher_model=teacher_model,
    )

    grads = None
    if use_wandb:
        if grokFast:
            wandb.init(project="grokking-universal", name=f"{selection_fn}_{'transformer' if is_transformer else 'mlp'}_grokfast_{seed}")
        else:
            wandb.init(project="grokking-universal", name=f"{selection_fn}_{'transformer' if is_transformer else 'mlp'}_{seed}")

    with tqdm(total=optimization_steps, desc="Model Training") as pbar:
        while step < optimization_steps:
            for datas in train_loader:
                if step >= optimization_steps:
                    break
                model.train()

                inputs = datas['input'].to(DEVICE)
                targets = datas['target'].to(DEVICE)
                indexes = datas.get('index', None)  # For selection methods that need sample indices

                # Online Batch Selection
                selected_inputs, selected_targets, selected_indexes = select_points(
                    inputs, targets, indexes, selector, selection_fn, fraction, model, is_transformer
                )

                # Forward pass - different for transformer vs MLP
                if is_transformer:
                    predictions, _ = model(selected_inputs)
                    loss = criterion(predictions[:, -1, :], selected_targets)  # Cross-entropy on last token
                else:
                    outputs = model(selected_inputs)
                    target = torch.nn.functional.one_hot(selected_targets, num_classes=10).float()
                    loss = criterion(outputs, target)  # MSE for MLP

                optimizer.zero_grad()
                loss.backward()

                if grokFast:
                    if is_transformer:
                        grads = gradfilter_ema(model, grads=grads, alpha=0.98, lamb=2.0)
                    else:
                        grads = gradfilter_ema(model, grads=grads, alpha=0.8, lamb=0.1)

                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

                if save_checkpoints and step % save_interval == 0:
                    model_type = "transformer" if is_transformer else "mlp"
                    torch.save(model.state_dict(), f"{selection_fn}_{model_type}_step_{step}.pth")

                # Calculate train statistics every log_interval
                if step % log_interval == 0:
                    train_acc, train_loss = test_train(model, train_loader, criterion, is_transformer)
                    val_acc, val_loss = test_val(model, val_loader, criterion, is_transformer)
                    train_accuracies.append(train_acc)
                    train_losses.append(train_loss)
                    val_accuracies.append(val_acc)
                    val_losses.append(val_loss)
                    if use_wandb:
                        wandb.log({
                            "train_accuracy": train_acc,
                            "train_loss": train_loss,
                            "val_accuracy": val_acc,
                            "val_loss": val_loss,
                            "step": step,
                            "batch_size": selected_inputs.shape[0],
                            "selection_fn": selection_fn,
                            "model_type": "transformer" if is_transformer else "mlp",
                            "lr": optimizer.param_groups[0]['lr'] if scheduler is not None else lr,
                            "weight_decay": weight_decay,
                            "grokFast": grokFast,
                            "seed": seed,
                        })

                step += 1
                pbar.update(1)

    if use_wandb:
        wandb.finish()

    return train_accuracies, train_losses, val_accuracies, val_losses

def test_train(model, train_loader, criterion, is_transformer):
    model.eval()
    all_preds = []
    all_labels = []
    epoch_loss = 0.0
    num_samples = 0
    with torch.no_grad():
        for data in train_loader:
            inputs = data['input'].to(DEVICE)
            targets = data['target'].to(DEVICE)

            if is_transformer:
                predictions, _ = model(inputs)
                loss = criterion(predictions[:, -1, :], targets)
                preds = torch.argmax(predictions[:, -1, :], dim=1)
            else:
                outputs = model(inputs)
                target = torch.nn.functional.one_hot(targets, num_classes=10).float()
                loss = criterion(outputs, target)
                preds = torch.argmax(outputs, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(targets.cpu().numpy())
            batch_size = targets.size(0)
            epoch_loss += loss.item() * batch_size
            num_samples += batch_size
    avg_train_loss = epoch_loss / num_samples
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    train_acc = np.mean(all_preds == all_labels)
    return train_acc, avg_train_loss

def test_val(model, val_loader, criterion, is_transformer):
    model.eval()
    all_preds = []
    all_labels = []
    epoch_loss = 0.0
    num_samples = 0
    with torch.no_grad():
        for data in val_loader:
            inputs = data['input'].to(DEVICE)
            targets = data['target'].to(DEVICE)

            if is_transformer:
                predictions, _ = model(inputs)
                loss = criterion(predictions[:, -1, :], targets)
                preds = torch.argmax(predictions[:, -1, :], dim=1)
            else:
                outputs = model(inputs)
                target = torch.nn.functional.one_hot(targets, num_classes=10).float()
                loss = criterion(outputs, target)
                preds = torch.argmax(outputs, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(targets.cpu().numpy())
            batch_size = targets.size(0)
            epoch_loss += loss.item() * batch_size
            num_samples += batch_size
    avg_epoch_test_loss = epoch_loss / num_samples
    all_preds =  np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    acc = np.mean(all_preds == all_labels)
    return acc, avg_epoch_test_loss

def select_points(inputs, targets, indexes, selector, selection_fn, fraction, model, is_transformer):
    n = inputs.shape[0]
    num_to_select = int(n * fraction)
    num_to_select = max(1, num_to_select)  # Ensure at least 1 sample is selected
    device = inputs.device

    if selection_fn == "full":
        indices = torch.arange(n, device=device)
        indices_np = indices.cpu().numpy()
    elif selection_fn == "uniform":
        indices_np = np.random.choice(n, num_to_select, replace=False)
    elif selection_fn == "loss_based":
        # Compute per-sample losses for selection
        model.eval()
        with torch.no_grad():
            if is_transformer:
                predictions, _ = model(inputs)
                per_sample_loss = torch.nn.functional.cross_entropy(
                    predictions[:, -1, :], targets, reduction='none'
                )
            else:
                outputs = model(inputs)
                target = torch.nn.functional.one_hot(targets, num_classes=10).float()
                per_sample_loss = torch.nn.functional.mse_loss(outputs, target, reduction='none').mean(dim=1)
        model.train()
        _, indices = torch.topk(per_sample_loss, num_to_select)
        indices_np = indices.cpu().numpy()
    elif selection_fn == "DivBS":
        if selector is None:
            indices_np = np.arange(n)
        else:
            indices_np = selector.select_points(inputs, targets, fraction=fraction)
    elif selection_fn == "RHOLoss":
        if selector is None:
            indices_np = np.arange(n)
        else:
            indices_np = selector.select_points(inputs, targets, indexes, num_to_select=num_to_select, epoch=0)
            indices_np = np.array(indices_np)
    elif selection_fn == "Bayesian":
        if selector is None:
            indices_np = np.arange(n)
        else:
            indices_np = selector.select_points(inputs, targets, indexes, num_to_select=num_to_select)
            indices_np = np.array(indices_np)
    else:
        raise ValueError(f"Unknown selection function: {selection_fn}")

    indices = torch.as_tensor(indices_np, device=device, dtype=torch.long)

    selected_inputs = inputs[indices]
    selected_targets = targets[indices]

    selected_indexes = None
    if indexes is not None:
        if not isinstance(indexes, torch.Tensor):
            indexes = torch.tensor(indexes, device=device)
        elif indexes.device != device:
            indexes = indexes.to(device)
        selected_indexes = indexes[indices]

    return selected_inputs, selected_targets, selected_indexes

def set_up_selector(model, selection_fn, is_transformer, train_loader=None, teacher_checkpoint_path=None, teacher_model=None):
    """Set up the appropriate selector based on selection method and model type.

    Args:
        model: The model to use for selection
        selection_fn: Selection method name ('DivBS', 'RHOLoss', 'Bayesian', etc.)
        is_transformer: Whether the model is a transformer
        train_loader: DataLoader for training data (required for some selectors)

    Returns:
        Selector object or None if selection_fn doesn't require a selector
    """
    train_dataset = getattr(train_loader, "dataset", None) if train_loader is not None else None

    if selection_fn == "DivBS":
        selector = DivBS(model, is_transformer=is_transformer)
    elif selection_fn == "RHOLoss":
        selector = RHOLoss(
            model,
            train_loader,
            is_transformer=is_transformer,
            teacher_checkpoint_path=teacher_checkpoint_path,
            teacher_model=teacher_model,
            uniform_epochs=0,
        )
    elif selection_fn == "Bayesian":
        selector = Bayesian(
            model,
            train_dataset=train_dataset,
            train_loader=train_loader,
            teacher_checkpoint_path=teacher_checkpoint_path,
            teacher_model=teacher_model,
            n_f_samples=256 #4 if is_transformer else 256,  # Fewer samples for transformers
        )
    else:
        selector = None

    return selector

In [6]:
# Experiments

def run_selection_methods(architecture, selection_methods, grokFast=False, seed=42):
    results = {}
    
    # Set seeds for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    for sel_fn in selection_methods:
        print(f"\n{'='*60}")
        print(f"Training with selection method: {sel_fn} @ grokFast={grokFast}")
        print(f"{'='*60}")

        # Set the seeds
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)

        if architecture == "MLP":
            model = SimpleMLP().to(DEVICE)
            weight_decay = 2.0 if grokFast else 0.01
            scale_factor = 8.0
            # model.apply(lambda m: scaled_kaiming_init(m, scale=scale_factor))
            with torch.no_grad():
                for p in model.parameters():
                    p.data = scale_factor * p.data
            criterion = torch.nn.MSELoss()
            teacher_checkpoint_path = None
            if sel_fn == "full":
                batch_size = 512  # Full batch for grokking
                current_fraction = 1.0
            else:
                batch_size = 250
                current_fraction = 0.

        elif architecture == "Transformer":
            model = GrokkModel(
                transformer_config=transformer_config,
                vocab_size=vocab_size,
                output_size=output_size,
                device=DEVICE
            ).to(DEVICE)
            weight_decay = 0.005 if grokFast else 0
            criterion = torch.nn.CrossEntropyLoss()
            teacher_checkpoint_path = "/home/phancock/Grokking/grokking_mnist/full_model_ModSubtract_GrokFast.pth"  # Use uncertainty-based selection
            if sel_fn == "full":
                batch_size = 128 # 128
                current_fraction = 1.0
            else:
                batch_size = 500
                current_fraction = 0.256
        else:
            raise ValueError(f"Architecture name {architecture} not valid. Expected MLP or Transformer.")

        lr = 1e-3
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.98))

        # Remove scheduler to allow immediate learning
        scheduler = None  # Removed warmup

        # Set up data depending on model architecture
        if architecture == "MLP":
            train_dataloader = torch.utils.data.DataLoader(wrapped_dataset(MNIST_train_data_subset), batch_size=batch_size, shuffle=False)
            val_dataloader = torch.utils.data.DataLoader(wrapped_dataset(MNIST_val_data), batch_size=batch_size, shuffle=False)
        else:
            train_dataloader = torch.utils.data.DataLoader(algorithmic_train_dataset, batch_size=batch_size, shuffle=False)
            val_dataloader = torch.utils.data.DataLoader(algorithmic_val_dataset, batch_size=batch_size, shuffle=False)

        optimization_steps = 100000
        log_interval = 500
        try:
            train_accs, train_losses, val_accs, val_losses = train(
                model, optimizer, criterion, train_dataloader, val_dataloader,
                optimization_steps=optimization_steps,
                log_interval=log_interval,
                save_checkpoints=False,
                save_interval=50000,
                selection_fn=sel_fn,
                fraction=current_fraction,
                use_wandb=True,
                grokFast=grokFast,
                teacher_checkpoint_path=teacher_checkpoint_path,
                scheduler=scheduler,
                lr=lr,
                weight_decay=weight_decay,
                seed=seed,
            )

            results[sel_fn] = {
                'train_accs': train_accs,
                'train_losses': train_losses,
                'val_accs': val_accs,
                'val_losses': val_losses
            }

            print(f"Final Results ({sel_fn}, grokFast={grokFast}):")
            print(f"  Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accs[-1]:.4f}")
            print(f"  Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}")
            plot_accuracies(train_accs, val_accs, log_interval, optimization_steps, include_ema=True, save_plot=False, plot_name=f"{sel_fn}_grokFast{grokFast}_accuracy_plot.png")
            plot_losses(train_losses, val_losses, log_interval, optimization_steps, save_plot=False, plot_name=f"{sel_fn}_grokFast{grokFast}_loss_plot.png")

        except Exception as e:
            print(f"Error training with {sel_fn}: {e}")
            results[sel_fn] = None

    print(f"\n{'='*60}")
    print(f"Training Comparison Complete for grokFast={grokFast}!")
    print(f"{'='*60}")

    print("\n" + "="*60)
    print(f"SUMMARY COMPARISON for grokFast={grokFast}")
    print("="*60)
    for method in selection_methods:
        if results.get(method) is not None:
            train_accs = results[method]['train_accs']
            val_accs = results[method]['val_accs']
            if len(train_accs) > 0 and len(val_accs) > 0:
                print(f"{method:12} | Train Acc: {train_accs[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}")
        else:
            print(f"{method:12} | Failed or skipped")

    return results

for architecture in ["Transformer"]:
    selection_methods = ["full, uniform, DivBS, RHOLoss"]
    # Run without grokFast first
    for i in range(3):
        results_no_grokfast = run_selection_methods(architecture, selection_methods, grokFast=False, seed=i)
    # Run with grokFast second
    for i in range(3):
        results_grokfast = run_selection_methods(architecture, selection_methods, grokFast=True, seed=i)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/phancock/.netrc.



Training with selection method: full, uniform, DivBS, RHOLoss @ grokFast=False


wandb: Currently logged in as: pjhancock2 (miller-ml-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run prd2iyce


wandb: Tracking run with wandb version 0.25.0


...

wandb: Syncing run full, uniform, DivBS, RHOLoss_transformer_grokfast_2


wandb: ⭐️ View project at https://wandb.ai/miller-ml-research/grokking-universal


wandb: 🚀 View run at https://wandb.ai/miller-ml-research/grokking-universal/runs/swaym5q2


Model Training:   0%|          | 0/100000 [00:00<?, ?it/s]

Error training with full, uniform, DivBS, RHOLoss: Unknown selection function: full, uniform, DivBS, RHOLoss

Training Comparison Complete for grokFast=True!

SUMMARY COMPARISON for grokFast=True
full, uniform, DivBS, RHOLoss | Failed or skipped


## Create Embeddings ##

In [7]:
# def create_embedding(model, model_name, train_data):
#     # Get features
#     X_features = model.stack[:-1](
#         train_data.data.view(len(train_data), -1).float()
#     ).detach().cpu().numpy()

#     # t-SNE
#     tsne = TSNE(n_components=2, random_state=42)
#     X_tsne = tsne.fit_transform(X_features)

#     print("t-SNE KL divergence:", tsne.kl_divergence_)

#     # Plot
#     fig = px.scatter(x=X_tsne[:, 0], y=X_tsne[:, 1], color=train_data.targets.numpy().astype(str), title="t-SNE Visualization of MNIST Features")
#     fig.update_layout(
#         title=f"t-SNE visualization of {model_name}",
#         xaxis_title="First t-SNE",
#         yaxis_title="Second t-SNE",
#     )
#     # fig.write_image(f"{model_name}_tsne.png")
#     fig.show()

# for i in [10000, 100000]:#range(0, 100000, 10000):
#     model_name = f"model_checkpoint_step_{i}.pth"
#     model = SimpleMLP()
#     model.load_state_dict(torch.load(f"mnist_checkpoints/model_checkpoint_step_{i}.pth"))
#     create_embedding(model, model_name, MNIST_train_data)